# Projeto 2

## Exercício 1

### Instalação de Bibliotecas 

In [3]:
!pip install pandas
!pip install numpy

  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/9.9 MB 812.7 kB/s eta 0:00:13
    --------------------------------------- 0.2/9.9 MB 2.0 MB/s eta 0:00:05
   -- ------------------------------------- 0.5/9.9 MB 3.3 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/9.9 MB 4.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.4/9.9 MB 5.7 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.9 MB 6.2 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.9 MB 6.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.9 MB 7.4 MB/s eta 0:00:01
   ------------- -------------------------- 3.4/9.9 MB 7.7 MB/s eta 0:00:01
   --------------- ------------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### QUESTION 0

Para começar iremos carregar um chunk pequeno para não estarmos a carregar o dataset inteiro de uma vez só.

In [6]:
import pandas as pd
import numpy as np
 
# Colunas value1 a value31
VALUE_COLS = [f'value{i}' for i in range(1, 32)]
USE_COLS   = ['id', 'year', 'month', 'element'] + VALUE_COLS
 
DATA_FILE     = 'data/ghcnd_daily.csv'
STATIONS_FILE = 'data/ghcnd-stations.txt'
CHUNK_SIZE    = 500_000

### ALÍNEA 1 

Leitura de um chunk e otimização de tipos de dados.

In [7]:
# Read only the first chunk; -9999 is treated as NaN
reader = pd.read_csv(
    DATA_FILE,
    usecols=USE_COLS,
    na_values=[-9999],
    chunksize=CHUNK_SIZE,
    low_memory=False
)
df = next(reader)
 
print(f"Chunk shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df.head(3))
 
# Memory usage before optimisation
mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory before optimisation: {mem_before:.2f} MB")
print(df.dtypes)
 
# Data type optimisation:
#   id, element  -> category  (repeated strings — large memory saving)
#   year         -> int16     (fits between -32768 and 32767)
#   month        -> int8      (1-12)
#   value1..31   -> float32   (sufficient precision for tenths of a degree)
df['id']      = df['id'].astype('category')
df['element'] = df['element'].astype('category')
df['year']    = df['year'].astype('int16')
df['month']   = df['month'].astype('int8')
for col in VALUE_COLS:
    df[col] = df[col].astype('float32')
 
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory after optimisation: {mem_after:.2f} MB")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")
print(df.dtypes)

Chunk shape: 500,000 rows x 35 columns
            id  year  month element  value1  value2  value3  value4  value5  \
0  ACW00011604  1949      1    TMAX   289.0   289.0   283.0   283.0   289.0   
1  ACW00011604  1949      2    TMAX   267.0   278.0   272.0   267.0   278.0   
2  ACW00011604  1949      3    TMAX   272.0   289.0   278.0   278.0   278.0   

   value6  ...  value22  value23  value24  value25  value26  value27  value28  \
0   289.0  ...    272.0    267.0    267.0    267.0    278.0    272.0    272.0   
1   278.0  ...    272.0    272.0    267.0    267.0    267.0    267.0    272.0   
2   278.0  ...    272.0    278.0    278.0    261.0    261.0    267.0    267.0   

   value29  value30  value31  
0    272.0    272.0    272.0  
1      NaN      NaN      NaN  
2    283.0    278.0    267.0  

[3 rows x 35 columns]

Memory before optimisation: 187.40 MB
id             str
year         int64
month        int64
element        str
value1     float64
value2     float64
value3     float64


### ALÍNEA 2

Percentagem de nulls por variável.

In [8]:
null_pct = (df.isnull().mean() * 100).round(2).rename('% nulls')
print("\nPercentage of null values per column:")
print(null_pct.to_string())


Percentage of null values per column:
id          0.00
year        0.00
month       0.00
element     0.00
value1      9.96
value2     10.00
value3      9.95
value4     10.08
value5      9.94
value6      9.95
value7      9.94
value8      9.82
value9      9.87
value10     9.93
value11     9.97
value12     9.96
value13     9.89
value14     9.94
value15    10.00
value16     9.90
value17     9.95
value18     9.99
value19     9.99
value20     9.89
value21     9.89
value22    10.00
value23     9.93
value24    10.49
value25    10.48
value26    10.32
value27    10.25
value28    10.08
value29    15.56
value30    17.48
value31    47.75


Como podemos observar as variáveis: 'value29, value30, value31' têm mais nulls porque nem todos os meses têm os dias 29, 30 e 31.

### Alínea 3

Ano mais velho e recente por cada estação.

In [9]:
year_range = (
    df.groupby('id', observed=True)['year']
    .agg(oldest_year='min', most_recent_year='max')
    .reset_index()
)
print(f"\nTotal stations in chunk: {len(year_range)}")
print(year_range.head(10))


Total stations in chunk: 1370
            id  oldest_year  most_recent_year
0  ACW00011604         1949              1949
1  ACW00011647         1961              1961
2  AE000041196         1944              2019
3  AEM00041194         1983              2019
4  AEM00041217         1983              2019
5  AEM00041218         1994              2019
6  AF000040930         1973              1992
7  AFM00040938         1973              2019
8  AFM00040948         1966              2019
9  AFM00040990         1973              2019


### Alínea 4

Temperatura média a cada dia por observação e criação da nova coluna -> daily_avg_temp.

In [10]:
# mean(axis=1) computes the average of value* columns for each row,
# automatically ignoring NaN values.
df['daily_avg_temp'] = df[VALUE_COLS].mean(axis=1)
 
print("\ndaily_avg_temp column created:")
print(df[['id', 'year', 'month', 'element', 'daily_avg_temp']].head(10))


daily_avg_temp column created:
            id  year  month element  daily_avg_temp
0  ACW00011604  1949      1    TMAX      274.612915
1  ACW00011604  1949      2    TMAX      271.142853
2  ACW00011604  1949      3    TMAX      277.935486
3  ACW00011604  1949      4    TMAX      287.166656
4  ACW00011604  1949      5    TMAX      291.354828
5  ACW00011604  1949      6    TMAX      294.833344
6  ACW00011604  1949      7    TMAX      298.709686
7  ACW00011647  1961     10    TMAX      272.000000
8  AE000041196  1944      3    TMAX      323.166656
9  AE000041196  1944      4    TMAX      321.466675


### Alínea 5

Média da coluna daily_avg_temp agrupados por estação e ano.

In [11]:
avg_by_station_year = (
    df.groupby(['id', 'year'], observed=True)['daily_avg_temp']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'daily_avg_temp': 'daily_avg_temp (yearly mean)'})
)
print(f"\nStation/year combinations: {len(avg_by_station_year):,}")
print(avg_by_station_year.head(10))


Station/year combinations: 44,364
            id  year  daily_avg_temp (yearly mean)
0  ACW00011604  1949                    285.109985
1  ACW00011647  1961                    272.000000
2  AE000041196  1944                    348.869995
3  AE000041196  1945                    318.230011
4  AE000041196  1955                    317.920013
5  AE000041196  1956                    318.109985
6  AE000041196  1957                    311.390015
7  AE000041196  1958                    317.899994
8  AE000041196  1959                    309.899994
9  AE000041196  1960                    316.929993


### Alínea 6

Dados das 5 estações meterológicas Portuguesas

In [14]:
# The ghcnd-stations.txt file has a fixed-width format:
# columns: ID (0-11), LAT (12-20), LON (21-30), ELEV (31-37),
#          STATE (38-40), NAME (41-71)
stations = pd.read_fwf(
    STATIONS_FILE,
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=['id', 'latitude', 'longitude', 'elevation', 'state', 'name'],
    encoding='utf-8'
)
print(f"\nTotal stations loaded: {len(stations):,}")
 
# Portuguese stations have the prefix 'PO'
portugal_names = ['HORTA', 'FUNCHAL', 'LISBOA', 'CASTELO BRANCO', 'FARO']
pt_stations = stations[
    stations['id'].str.startswith('PO') &
    stations['name'].str.upper().str.contains('|'.join(portugal_names))
].copy()
 
print("\nPortuguese stations found:")
print(pt_stations[['id', 'name']].to_string(index=False))
 
# Filter the chunk by Portuguese station IDs
pt_ids = pt_stations['id'].tolist()
df_portugal = df[df['id'].isin(pt_ids)][USE_COLS].copy()
 
print(f"\nRecords found in chunk: {len(df_portugal):,}")
print(df_portugal.head(10))


Total stations loaded: 114,789

Portuguese stations found:
         id               name
PO000008506     HORTA (AZORES)
PO000008522            FUNCHAL
PO000008535   LISBOA GEOFISICA
POM00008521 FUNCHAL/S.CATARINA
POM00008554               FARO
POM00008570     CASTELO BRANCO

Records found in chunk: 0
Empty DataFrame
Columns: [id, year, month, element, value1, value2, value3, value4, value5, value6, value7, value8, value9, value10, value11, value12, value13, value14, value15, value16, value17, value18, value19, value20, value21, value22, value23, value24, value25, value26, value27, value28, value29, value30, value31]
Index: []

[0 rows x 35 columns]


Como os IDs das estações portuguesas normalmente começam por "PO", elas não aparecem nas primeiras 500.000 linhas do ficheiro. Por isso, o df_portugal ficou vazio.

Assim para contornar-mos esse mini-problema corremos o ficheiro inteiro (todos os chunks) e guardamos apenas as linhas que nos interessam.

In [ ]:
# Load the stations information to find the IDs for the requested cities
# Portuguese stations names: Horta, Funchal, Lisboa, Castelo Branco, Faro
portugal_target_names = ['HORTA', 'FUNCHAL', 'LISBOA', 'CASTELO BRANCO', 'FARO']

# The stations file is Fixed-Width Format (FWF) 
stations = pd.read_fwf(
    STATIONS_FILE,
    colspecs=[(0, 11), (41, 71)],
    names=['id', 'name'],
    encoding='utf-8'
)

# Filter the stations that are Portuguese (start with 'PO') and match our target names 
pt_stations = stations[
    stations['id'].str.startswith('PO') & 
    stations['name'].str.upper().str.contains('|'.join(portugal_target_names))
].copy()

pt_ids = pt_stations['id'].tolist()
print(f"IDs found for Portuguese stations: {pt_ids}")

# Process the ENTIRE dataset in chunks to collect all Portuguese records
portugal_chunks = []

# Re-initialize the reader to start from the beginning of the file
reader = pd.read_csv(
    DATA_FILE,
    usecols=USE_COLS,
    na_values=[-9999],
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for chunk in reader:
    # Filter the current chunk for the identified Portuguese IDs 
    mask = chunk['id'].isin(pt_ids)
    pt_data = chunk[mask].copy()
    
    if not pt_data.empty:
        portugal_chunks.append(pt_data)

# Concatenate all found records into a single DataFrame
df_portugal = pd.concat(portugal_chunks, ignore_index=True)

print(f"\nTotal records found for Portugal: {len(df_portugal):,}")
print(df_portugal.head())

IDs found for Portuguese stations: ['PO000008506', 'PO000008522', 'PO000008535', 'POM00008521', 'POM00008554', 'POM00008570']

Total records found for Portugal: 3,161
            id  year  month element  value1  value2  value3  value4  value5  \
0  PO000008506  1973      1    TMAX     NaN   110.0     NaN   170.0   160.0   
1  PO000008506  1973      2    TMAX   170.0   160.0   120.0   110.0   130.0   
2  PO000008506  1973      3    TMAX     NaN   150.0   160.0     NaN   160.0   
3  PO000008506  1973      4    TMAX   180.0   170.0     NaN   190.0   170.0   
4  PO000008506  1973      5    TMAX   140.0     NaN     NaN   150.0     NaN   

   value6  ...  value22  value23  value24  value25  value26  value27  value28  \
0   160.0  ...      NaN      NaN      NaN    130.0      NaN      NaN      NaN   
1     NaN  ...      NaN    150.0      NaN    170.0    180.0    170.0    170.0   
2     NaN  ...      NaN      NaN    160.0    170.0    170.0    180.0      NaN   
3     NaN  ...      NaN    160.0  

### Alínea 7

Troca dos IDs pelo nome das estações metereológicas correspondentes.

In [17]:
id_to_name = dict(zip(pt_stations['id'], pt_stations['name'].str.title()))
print("\nID -> Name mapping:")
for k, v in id_to_name.items():
    print(f"  {k}  ->  {v}")
 
df_portugal = df_portugal.copy()
df_portugal['id'] = df_portugal['id'].map(id_to_name)
df_portugal = df_portugal.rename(columns={'id': 'station_name'})
 
print("\nFinal DataFrame with station names:")
print(df_portugal[['station_name', 'year', 'month', 'element']].head(15))
 
print("\nRecord count per station:")
print(df_portugal['station_name'].value_counts().to_string())


ID -> Name mapping:
  PO000008506  ->  Horta (Azores)
  PO000008522  ->  Funchal
  PO000008535  ->  Lisboa Geofisica
  POM00008521  ->  Funchal/S.Catarina
  POM00008554  ->  Faro
  POM00008570  ->  Castelo Branco

Final DataFrame with station names:
      station_name  year  month element
0   Horta (Azores)  1973      1    TMAX
1   Horta (Azores)  1973      2    TMAX
2   Horta (Azores)  1973      3    TMAX
3   Horta (Azores)  1973      4    TMAX
4   Horta (Azores)  1973      5    TMAX
5   Horta (Azores)  1973      6    TMAX
6   Horta (Azores)  1973      7    TMAX
7   Horta (Azores)  1973      8    TMAX
8   Horta (Azores)  1973      9    TMAX
9   Horta (Azores)  1973     10    TMAX
10  Horta (Azores)  1973     11    TMAX
11  Horta (Azores)  1973     12    TMAX
12  Horta (Azores)  1974      1    TMAX
13  Horta (Azores)  1974      2    TMAX
14  Horta (Azores)  1974      3    TMAX

Record count per station:
station_name
Lisboa Geofisica      1377
Faro                   537
Funchal/S.Catar

## Exercício 2